# Sweep Summary
Loads `sweep_summary.csv` and generates plots for all runs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf as pdf_backend
import seaborn as sns
from pathlib import Path

# ── Config ──────────────────────────────────────────────────────────
SUMMARY_CSV = Path('reports/sweep/sweep_summary.csv') 
PDF_OUT     = Path('reports/sweep_summary_report.pdf')
# ────────────────────────────────────────────────────────────────────

# DTU colors
DTU_RED = (0.6, 0, 0)
DTUBLUE = (0.1843, 0.2431, 0.9176)
DTUBRIGHTGREEN = (0.1216, 0.8157, 0.5098)
DTUNAVY = (0.0118, 0.0588, 0.3098)
DTUYELLOW = (0.9647, 0.8157, 0.3019)
DTUORANGE = (0.9882, 0.4627, 0.2039)
DTUPINK = (0.9686, 0.7333, 0.6941)
DTUGREY = (0.8549, 0.8549, 0.8549)
DTURED = (0.9098, 0.2471, 0.2824)
DTUGREEN = (0, 0.5333, 0.2078)
DTUPURPLE = (0.4745, 0.1373, 0.5569)

# Colors for each model (using DTU colors)
COLORS = {
    'Random Forest':      DTUBLUE,
    'Logistic Regression':DTU_RED,
    'SVM':                DTUORANGE,
    'XGBoost':            DTUPURPLE,
}


df = pd.read_csv(SUMMARY_CSV)
print(f'Loaded {len(df)} runs from {SUMMARY_CSV}')
df.head()

## 1 — Runs summary table

In [ ]:
cols = ['run', 'seed', 'test_size', 'mode', 'binary_mode', 'best_model',
        'cv_bal_acc', 'cv_bal_std', 'test_bal_acc', 'beats_baseline', 'status']
display(df[cols].style
    .format({'cv_bal_acc': '{:.3f}', 'cv_bal_std': '{:.3f}', 'test_bal_acc': '{:.3f}'})
    .applymap(lambda v: 'color: green' if v is True else ('color: red' if v is False else ''),
              subset=['beats_baseline'])
)

## 2 — Test balanced accuracy by mode & binary_mode

In [ ]:
ok = df[df['status'] == 'ok'].copy()
ok['label'] = ok.apply(
    lambda r: r['binary_mode'] if r['mode'] == 'binary' else 'multiclass', axis=1
)

fig, ax = plt.subplots(figsize=(10, 5))
labels  = ok['label'].unique()
x       = np.arange(len(labels))
models  = ok['best_model'].unique()
width   = 0.8 / len(models)

for i, model in enumerate(models):
    sub    = ok[ok['best_model'] == model]
    means  = [sub[sub['label'] == l]['test_bal_acc'].mean() for l in labels]
    stds   = [sub[sub['label'] == l]['cv_bal_std'].mean()   for l in labels]
    offset = (i - len(models) / 2 + 0.5) * width
    ax.bar(x + offset, means, width, yerr=stds, label=model,
           color=COLORS.get(model, 'grey'), alpha=0.85,
           error_kw=dict(elinewidth=0.8, capsize=3))

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Test balanced accuracy', fontsize=10)
ax.set_title('Test balanced accuracy by classification mode', fontsize=12,
             fontweight='bold', color=DTUNAVY)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(0, 1)
plt.tight_layout()
fig_bar = fig
plt.show()

## 3 — Overfitting flags (cv_test_diff)

In [ ]:
OVERFIT_THRESHOLD = 0.1  # flag if CV acc - test acc > threshold

flag_cols = ['run', 'seed', 'test_size', 'mode', 'binary_mode',
             'best_model', 'cv_bal_acc', 'test_bal_acc', 'cv_test_diff']

def _flag(v):
    if isinstance(v, float) and v > OVERFIT_THRESHOLD:
        return 'background-color: #ffcccc'
    return ''

display(ok[flag_cols].style
    .format({'cv_bal_acc': '{:.3f}', 'test_bal_acc': '{:.3f}', 'cv_test_diff': '{:.3f}'})
    .applymap(_flag, subset=['cv_test_diff'])
)

# Also plot
fig, ax = plt.subplots(figsize=(10, 4))
colors  = [DTURED if v > OVERFIT_THRESHOLD else DTUNAVY for v in ok['cv_test_diff']]
ax.bar(ok['run'], ok['cv_test_diff'], color=colors, alpha=0.8)
ax.axhline(OVERFIT_THRESHOLD, color=DTURED, linestyle='--', linewidth=0.8,
           label=f'Threshold ({OVERFIT_THRESHOLD})')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Run', fontsize=10)
ax.set_ylabel('CV acc − Test acc', fontsize=10)
ax.set_title('Overfitting check (cv_test_diff per run)', fontsize=12,
             fontweight='bold', color=DTUNAVY)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
fig_overfit = fig
plt.show()

## 4 — McNemar results

In [ ]:
mcnemar_p_cols     = [c for c in ok.columns if c.startswith('mcnemar_p_')]
mcnemar_theta_cols = [c for c in ok.columns if c.startswith('mcnemar_theta_')]

id_cols = ['run', 'mode', 'binary_mode', 'best_model']

if mcnemar_p_cols:
    display(ok[id_cols + mcnemar_p_cols + mcnemar_theta_cols].style
        .format({c: '{:.3f}' for c in mcnemar_p_cols + mcnemar_theta_cols})
        .applymap(lambda v: 'background-color: #ccffcc' if isinstance(v, float) and v < 0.05 else '',
                  subset=mcnemar_p_cols)
    )

    # Heatmap of p-values
    pivot = ok.set_index('run')[mcnemar_p_cols]
    pivot.columns = [c.replace('mcnemar_p_', '').replace('_vs_best', '') for c in pivot.columns]

    fig, ax = plt.subplots(figsize=(max(6, len(pivot.columns) * 2), max(4, len(pivot) * 0.4 + 1)))
    sns.heatmap(pivot.astype(float), annot=True, fmt='.3f', cmap='RdYlGn_r',
                vmin=0, vmax=1, linewidths=0.4, ax=ax,
                cbar_kws={'label': 'p-value'})
    ax.set_title('McNemar p-values (best model vs others)', fontsize=12,
                 fontweight='bold', color=DTUNAVY)
    ax.set_xlabel('Compared model', fontsize=10)
    ax.set_ylabel('Run', fontsize=10)
    plt.tight_layout()
    fig_mcnemar = fig
    plt.show()
else:
    print('No McNemar columns found in summary.')
    fig_mcnemar = None

## 5 — Save to PDF

In [ ]:
import datetime

PDF_OUT.parent.mkdir(parents=True, exist_ok=True)

figs = [fig_bar, fig_overfit]
if fig_mcnemar is not None:
    figs.append(fig_mcnemar)

with pdf_backend.PdfPages(PDF_OUT) as pdf:
    # ── Cover page ──────────────────────────────────────────────────
    fig_cover = plt.figure(figsize=(12, 6))
    fig_cover.patch.set_facecolor('white')
    fig_cover.text(0.5, 0.65, 'Sweep Summary Report',
                   ha='center', fontsize=20, fontweight='bold', color=DTUNAVY)
    fig_cover.text(0.5, 0.52, f'Source: {SUMMARY_CSV}',
                   ha='center', fontsize=10, color='grey')
    fig_cover.text(0.5, 0.44,
                   datetime.datetime.now().strftime('Generated %Y-%m-%d  %H:%M'),
                   ha='center', fontsize=10, color='grey')
    fig_cover.text(0.5, 0.35, f'{len(ok)} runs  |  {ok["mode"].unique().tolist()}',
                   ha='center', fontsize=10, color=DTUNAVY)
    pdf.savefig(fig_cover, bbox_inches='tight')
    plt.close(fig_cover)

    # ── Runs table page ─────────────────────────────────────────────
    tbl_cols = ['run', 'seed', 'test_size', 'mode', 'binary_mode',
                'best_model', 'cv_bal_acc', 'cv_bal_std', 'test_bal_acc',
                'beats_baseline', 'status']
    tbl_data = ok[tbl_cols].round(3)
    fig_tbl, ax_tbl = plt.subplots(figsize=(16, max(4, len(tbl_data) * 0.4 + 1)))
    ax_tbl.axis('off')
    tbl = ax_tbl.table(
        cellText=tbl_data.values,
        colLabels=tbl_cols,
        cellLoc='center', loc='center'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.auto_set_column_width(col=list(range(len(tbl_cols))))
    ax_tbl.set_title('All runs', fontsize=12, fontweight='bold',
                     color=DTUNAVY, pad=12)
    pdf.savefig(fig_tbl, bbox_inches='tight')
    plt.close(fig_tbl)

    # ── Plot pages ───────────────────────────────────────────────────
    for fig in figs:
        pdf.savefig(fig, bbox_inches='tight')

    meta = pdf.infodict()
    meta['Title']  = 'Sweep Summary Report'
    meta['Author'] = 'EOG_REM pipeline'

print(f'Saved → {PDF_OUT}')